In [ ]:
import os

# Update you config directory as needed
os.environ["PSYNC_CONFIG_DIR"] = "../config"

import logging

logging.basicConfig(level="DEBUG")

# Sync example Tidal and Spotify

In this example we will show how to sync a playlist from Spotify to Tidal. This is a pretty easy use case, because tracks on both services can be identified by their [ISRC](https://en.wikipedia.org/wiki/International_Standard_Recording_Code).

To execute this example you will need to configure both the spotify and tidal services.


## The initial playlist

We start with a spotify playlist here and create a tidal playlist with the same name if it does not exist yet. You can
also start with two blank playlists.


In [ ]:
from plistsync.services.spotify import (
    SpotifyLibraryCollection,
)

spotify_library = SpotifyLibraryCollection()

# Spotify id of the playlist to sync (Drum & Bass Top 100)
# By name
if spotify_playlist := spotify_library.get_playlist(
    url="https://open.spotify.com/playlist/0Zarq4BVkFkZOWkmqsfrjA"
):
    print(
        f"Found playlist: {spotify_playlist.name} "
        f"({len(spotify_playlist.tracks)} tracks)"
    )
else:
    print("Playlist not found.")

In [ ]:
spotify_playlist.name

Next up we create a corresponding playlist on Tidal (or get it if it already exists):

In [ ]:
from plistsync.services.tidal import TidalLibraryCollection, TidalPlaylistCollection

tidal_library = TidalLibraryCollection()

if tidal_library.has_playlist(spotify_playlist.name):
    # Playlist exists, we can use it
    _tidal_playlist = tidal_library.get_playlist(name=spotify_playlist.name)
    assert _tidal_playlist is not None
    tidal_playlist = _tidal_playlist
    print(
        f"Using existing Tidal playlist '{tidal_playlist.id}' "
        f"with {len(tidal_playlist)} tracks"
    )
else:
    # Create the playlist
    resource, lookup = tidal_library.api.playlist.create(
        name=spotify_playlist.name,
        description=spotify_playlist.description or "",
    )
    tidal_playlist = TidalPlaylistCollection.from_response_data(
        tidal_library, resource, lookup
    )
    print(f"Created Tidal playlist '{tidal_playlist.id}'")

We now have both playlists and can start to sync them. We will add tracks that are in the Spotify playlist but not in the Tidal one.

The following might not be super efficient, but it works for demonstration purposes. Maybe we can optimize it later?

In [ ]:
from plistsync.services.spotify import SpotifyPlaylistTrack
from plistsync.services.tidal import TidalPlaylistTrack

# Find tracks in Spotify playlist that are not in Tidal playlist
missing_tracks: list[SpotifyPlaylistTrack] = []

tidal_playlist._refetch_tracks()
print([t for t in tidal_playlist.tracks])

for track_s in spotify_playlist.tracks:
    # Try to match by ISRC only (cutoff 1.0)
    # TODO: implement playlist.match logic (global and local lookup, so match works)
    # matches = tidal_playlist.match(track_s, cutoff=0.8)
    # if len(matches.found) == 0:
    #     missing_tracks.append(track_s)

    if track_s.isrc is not None and track_s.isrc not in [
        t.isrc for t in tidal_playlist.tracks
    ]:
        missing_tracks.append(track_s)

print(
    f"Found {len(missing_tracks)} missing tracks in Tidal playlist '{tidal_playlist.name}'"
)


tidal_tracks = tidal_library.find_many_by_global_ids(
    map(lambda t: t.global_ids, missing_tracks)
)
tidal_tracks = list(filter(None, tidal_tracks))
print(f"Found {len(tidal_tracks)} out of {len(missing_tracks)} tracks on Tidal")

# service-level tracks have less information than playlist tracks, so we have to convert
tidal_tracks = [TidalPlaylistTrack(t) for t in tidal_tracks]

with tidal_playlist.remote_edit():
    # as of now, inserts are one api call each. (this will take a while)
    tidal_playlist.tracks.extend(tidal_tracks)

print(f"Tidal playlist '{tidal_playlist.name}' now has {len(tidal_playlist)} tracks")

One directional sync is now done. More features coming soon!